In [ ]:
import sys
!{sys.executable} -m pip install pandas yfinance vaderSentiment statsmodels scipy matplotlib seaborn

In [ ]:
import pandas as pd
import yfinance as yf

In [ ]:
tweets = pd.read_csv('../data/raw/tweets.csv')
print(tweets.shape)
print(tweets.head())
print(tweets.columns.tolist())

In [3]:
# Defense-related keywords
keywords = [
    'defense', 'military', 'NATO', 'army', 'weapons', 'war',
    'troops', 'missile', 'nuclear', 'China', 'Iran', 'threat',
    'bomb', 'attack', 'soldiers', 'Pentagon', 'armed forces'
]

pattern = '|'.join(keywords)
defense_tweets = tweets[tweets['text'].str.contains(pattern, case=False, na=False)]

print(f"Total tweets: {len(tweets)}")
print(f"Defense-related tweets: {len(defense_tweets)}")
defense_tweets[['date', 'text']].head(10)

Total tweets: 56571
Defense-related tweets: 5850


,date,text
3,2020-09-12 20:10:58,The Unsolicited Mail In Ballot Scam is a major...
20,2020-10-12 22:22:39,"“I’m running as a proud Democrat, for the Sena..."
50,2021-01-04 15:23:50,“We are not acting to thwart the Democratic pr...
64,2020-10-16 11:15:55,RT @RandPaul: Somebody is mad that @realDonald...
71,2020-10-16 11:01:28,"RT @RandPaul: In his term, @realDonaldTrump ha..."
92,2020-11-16 13:27:55,RT @realDonaldTrump: ANTIFA SCUM ran for the h...
102,2021-01-05 14:50:49,Pleased to announce that @KLoeffler &amp; @sen...
106,2020-11-16 14:19:29,Another Vaccine just announced. This time by M...
114,2021-01-05 15:02:22,"Georgia, get out and VOTE for two great Senato..."
158,2021-01-05 22:25:08,"Antifa is a Terrorist Organization, stay out o..."


In [4]:
# Parse dates and filter to presidential term
defense_tweets = defense_tweets.copy()
defense_tweets['date'] = pd.to_datetime(defense_tweets['date'])

# Keep only presidential term: Jan 20, 2017 – Jan 20, 2021
mask = (defense_tweets['date'] >= '2017-01-20') & (defense_tweets['date'] <= '2021-01-20')
defense_tweets = defense_tweets[mask]

print(f"Tweets after date filter: {len(defense_tweets)}")
print(f"Date range: {defense_tweets['date'].min()} to {defense_tweets['date'].max()}")

Tweets after date filter: 3417
Date range: 2017-01-24 17:49:17 to 2021-01-05 22:25:08


In [5]:
# Define tickers - US and EU defense companies
tickers_us = ['LMT', 'RTX', 'NOC', 'BA', 'GD']
tickers_eu = ['AIR.PA', 'BA.L', 'HO.PA', 'SAF.PA', 'RHM.DE']
all_tickers = tickers_us + tickers_eu

# Download adjusted closing prices
prices = yf.download(
    all_tickers,
    start='2017-01-01',
    end='2021-01-25',
    auto_adjust=True
)['Close']

print(f"Shape: {prices.shape}")
print(f"Missing values:\n{prices.isnull().sum()}")
prices.head()

[*********************100%***********************]  10 of 10 completed

Shape: (1049, 10)
Missing values:
Ticker
AIR.PA    11
BA        28
BA.L      22
GD        28
HO.PA     11
LMT       28
NOC       28
RHM.DE    22
RTX       28
SAF.PA    11
dtype: int64


Ticker,AIR.PA,BA,BA.L,GD,HO.PA,LMT,NOC,RHM.DE,RTX,SAF.PA
Date,,,,,,,,,,
2017-01-02,55.855190,NaN,NaN,NaN,77.609055,NaN,NaN,54.837006,NaN,63.202072
2017-01-03,56.459801,145.533691,589.186218,143.258774,76.620140,198.397354,202.550125,55.279102,56.299606,63.358856
2017-01-04,56.370888,147.063461,583.204651,143.332306,76.307404,198.929916,202.860565,54.522430,56.335178,63.183620
2017-01-05,57.437840,147.146927,585.696838,143.822205,75.977760,200.488541,203.498718,54.845509,56.563766,62.491943
2017-01-06,57.917973,147.508484,603.143250,144.899948,75.673485,201.953217,204.136795,55.168579,57.173344,62.436596


In [6]:
# Forward fill missing values (different market holidays)
prices_filled = prices.ffill()

# Calculate daily log returns
returns = prices_filled.pct_change().dropna()

print(f"Returns shape: {returns.shape}")
print(f"Date range: {returns.index[0]} to {returns.index[-1]}")
returns.head()

Returns shape: (1047, 10)
Date range: 2017-01-04 00:00:00 to 2021-01-22 00:00:00


Ticker,AIR.PA,BA,BA.L,GD,HO.PA,LMT,NOC,RHM.DE,RTX,SAF.PA
Date,,,,,,,,,,
2017-01-04,-0.001575,0.010511,-0.010152,0.000513,-0.004082,0.002684,0.001533,-0.013688,0.000632,-0.002766
2017-01-05,0.018927,0.000568,0.004273,0.003418,-0.004320,0.007835,0.003146,0.005926,0.004058,-0.010947
2017-01-06,0.008359,0.002457,0.029787,0.007494,-0.004005,0.007306,0.003136,0.005891,0.010777,-0.000886
2017-01-09,0.002149,-0.004903,-0.012397,-0.007832,0.005250,-0.000155,-0.003886,-0.010171,-0.009329,-0.001329
2017-01-10,-0.001226,0.004737,0.020921,0.001647,0.000333,-0.006361,-0.011450,0.000311,-0.002152,-0.004141


In [7]:
# Save processed data
defense_tweets.to_csv('../data/processed/tweets_filtered.csv', index=False)
returns.to_csv('../data/processed/returns.csv')

print("Data saved successfully.")
print(f"Tweets: {len(defense_tweets)} rows")
print(f"Returns: {returns.shape[0]} rows x {returns.shape[1]} tickers")

Data saved successfully.
Tweets: 3417 rows
Returns: 1047 rows x 10 tickers
